# RuleShift CogFlex Benchmark

Multi-turn cognitive-flexibility benchmark focused on switching between active classification rules.

## Notebook Map

1. Bootstrap dataset selection
2. Define runtime types and parsers
3. Normalize model responses
4. Run the multi-turn episode scorer
5. Load the frozen benchmark rows
6. Register the official Kaggle task
7. Build optional diagnostics
8. Mark the official entry point

In [ ]:
import os
from pathlib import Path

import kaggle_benchmarks as kbench
import pandas as pd

DATASET_ROOT_ENV_VAR = "RULESHIFT_DATASET_ROOT"
PRIVATE_DATASET_ROOT_ENV_VAR = "RULESHIFT_PRIVATE_DATASET_ROOT"
PRIVATE_ANSWER_KEY_PATH_ENV_VAR = "RULESHIFT_PRIVATE_ANSWER_KEY_PATH"
EVAL_SPLIT = os.environ.get("RULESHIFT_EVAL_SPLIT", "public").strip().lower()
EXPECTED_PUBLIC_EPISODE_COUNT = 80
EXPECTED_PRIVATE_EPISODE_COUNT = 400
EXPECTED_EPISODES_PER_GROUP = {"public": 20, "private": 100}

PUBLIC_ROWS_FILENAME = "public_leaderboard_rows.json"
PRIVATE_ROWS_FILENAME = "private_leaderboard_rows.json"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

def _dataset_roots(dataset_slug: str) -> tuple[Path, ...]:
    return (
        KAGGLE_INPUT_ROOT / "datasets" / "raptorengineer" / dataset_slug,
        KAGGLE_INPUT_ROOT / dataset_slug,
        KAGGLE_INPUT_ROOT / "raptorengineer" / dataset_slug,
        KAGGLE_INPUT_ROOT / f"raptorengineer-{dataset_slug}",
    )

def _resolve_rows_path(
    split: str,
    filename: str,
    explicit_root: Path | None,
    default_roots: tuple[Path, ...],
    search_roots: tuple[Path, ...] = (KAGGLE_INPUT_ROOT,),
) -> Path:
    candidates: list[Path] = []
    if explicit_root is not None:
        candidates.append(explicit_root / filename)
    else:
        candidates.extend(root / filename for root in default_roots)
        for search_root in search_roots:
            if not search_root.exists():
                continue
            candidates.extend(sorted(search_root.glob(f"*/{filename}")))
            candidates.extend(sorted(search_root.glob(f"*/*/{filename}")))

    seen: set[Path] = set()
    unique_candidates: list[Path] = []
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        unique_candidates.append(candidate)
        if candidate.exists():
            return candidate

    checked = ", ".join(str(path) for path in unique_candidates) or "<none>"
    raise FileNotFoundError(f"Selected split rows file not found for split {split!r}. Checked: {checked}")

DEFAULT_DATASET_ROOT = _dataset_roots("ruleshift-cogflex-runtime")[0]
DEFAULT_PRIVATE_DATASET_ROOT = _dataset_roots("ruleshift-cogflex-runtime-private")[0]
_DATASET_ROOT_OVERRIDE = os.environ.get(DATASET_ROOT_ENV_VAR, "").strip()
_PRIVATE_DATASET_ROOT_OVERRIDE = os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR, "").strip()
DATASET_ROOT = Path(_DATASET_ROOT_OVERRIDE) if _DATASET_ROOT_OVERRIDE else DEFAULT_DATASET_ROOT
PRIVATE_DATASET_ROOT = Path(_PRIVATE_DATASET_ROOT_OVERRIDE) if _PRIVATE_DATASET_ROOT_OVERRIDE else DEFAULT_PRIVATE_DATASET_ROOT
_PRIVATE_ANSWER_KEY_PATH = os.environ.get(PRIVATE_ANSWER_KEY_PATH_ENV_VAR, "").strip()
PRIVATE_ANSWER_KEY_PATH = Path(_PRIVATE_ANSWER_KEY_PATH) if _PRIVATE_ANSWER_KEY_PATH else None

if EVAL_SPLIT not in {"public", "private"}:
    raise ValueError(f"Unsupported EVAL_SPLIT: {EVAL_SPLIT!r}")

if EVAL_SPLIT == "public":
    ROWS_PATH = _resolve_rows_path(
        split=EVAL_SPLIT,
        filename=PUBLIC_ROWS_FILENAME,
        explicit_root=Path(_DATASET_ROOT_OVERRIDE) if _DATASET_ROOT_OVERRIDE else None,
        default_roots=_dataset_roots("ruleshift-cogflex-runtime"),
    )
    DATASET_ROOT = ROWS_PATH.parent
else:
    ROWS_PATH = _resolve_rows_path(
        split=EVAL_SPLIT,
        filename=PRIVATE_ROWS_FILENAME,
        explicit_root=Path(_PRIVATE_DATASET_ROOT_OVERRIDE) if _PRIVATE_DATASET_ROOT_OVERRIDE else None,
        default_roots=_dataset_roots("ruleshift-cogflex-runtime-private"),
    )
    PRIVATE_DATASET_ROOT = ROWS_PATH.parent


In [ ]:
import re
from collections import Counter
from dataclasses import dataclass
from enum import Enum
from typing import Final

class Label(str, Enum):
    type_a = "type_a"
    type_b = "type_b"

PROBE_COUNT: Final[int] = 4
TURN_COUNT: Final[int] = 3
FACULTY_ID: Final[str] = "executive_functions/cognitive_flexibility"
ALLOWED_GROUPS: Final[frozenset[str]] = frozenset({"explicit_switch", "reversal", "latent_switch", "context_switch"})
CONTEXTS: Final[tuple[str, str]] = ("alpha", "beta")
OUTPUT_INSTRUCTION: Final[str] = "Return exactly 4 outputs in order, one per probe. Use only type_a or type_b."
TURN_HEADER_PREFIX: Final[str] = "RuleShift cognitive flexibility task. Episode "
GROUP_SHIFT_MODES: Final[dict[str, str]] = {
    "explicit_switch": "explicit_instruction",
    "reversal": "label_reversal",
    "latent_switch": "latent_example_change",
    "context_switch": "context_gate",
}
EXAMPLE_LINE_RE: Final[re.Pattern[str]] = re.compile(
    r"^(?P<index>\d+)\.\s+(?:(?:context)=(?P<context>[a-z]+)\s+\|\s+)?r1=(?P<r1>[+-]\d+),\s+r2=(?P<r2>[+-]\d+)\s+->\s+(?P<label>type_a|type_b)$"
)
PROBE_LINE_RE: Final[re.Pattern[str]] = re.compile(
    r"^(?P<index>\d+)\.\s+(?:(?:context)=(?P<context>[a-z]+)\s+\|\s+)?r1=(?P<r1>[+-]\d+),\s+r2=(?P<r2>[+-]\d+)\s+->\s+\?$"
)
PROBE_FIELDS: Final[tuple[str, str, str, str]] = ("probe_1", "probe_2", "probe_3", "probe_4")
_ALLOWED_LABELS: Final[frozenset[str]] = frozenset(label.value for label in Label)
RULES: Final[dict[str, object]] = {
    "r1_at_least_minus_1": lambda r1, r2: r1 >= -1,
    "r1_at_least_plus_2": lambda r1, r2: r1 >= 2,
    "r2_at_least_minus_1": lambda r1, r2: r2 >= -1,
    "r2_at_least_plus_2": lambda r1, r2: r2 >= 2,
    "r1_at_most_minus_1": lambda r1, r2: r1 <= -1,
    "r1_at_most_plus_1": lambda r1, r2: r1 <= 1,
    "r2_at_most_minus_1": lambda r1, r2: r2 <= -1,
    "r2_at_most_plus_1": lambda r1, r2: r2 <= 1,
    "sum_at_least_plus_1": lambda r1, r2: r1 + r2 >= 1,
    "sum_at_least_plus_4": lambda r1, r2: r1 + r2 >= 4,
    "sum_at_most_minus_1": lambda r1, r2: r1 + r2 <= -1,
    "sum_at_most_minus_4": lambda r1, r2: r1 + r2 <= -4,
    "diff_at_least_plus_1": lambda r1, r2: r1 - r2 >= 1,
    "diff_at_least_plus_4": lambda r1, r2: r1 - r2 >= 4,
    "diff_at_most_minus_1": lambda r1, r2: r1 - r2 <= -1,
    "diff_at_most_minus_4": lambda r1, r2: r1 - r2 <= -4,
    "abs_r1_at_least_2": lambda r1, r2: abs(r1) >= 2,
    "abs_r2_at_least_2": lambda r1, r2: abs(r2) >= 2,
    "same_sign": lambda r1, r2: (r1 > 0) == (r2 > 0),
    "r1_positive": lambda r1, r2: r1 > 0,
    "abs_equal": lambda r1, r2: abs(r1) == abs(r2),
    "abs_r1_greater_than_abs_r2": lambda r1, r2: abs(r1) > abs(r2),
    "abs_r1_less_than_abs_r2": lambda r1, r2: abs(r1) < abs(r2),
    "both_positive": lambda r1, r2: r1 > 0 and r2 > 0,
    "both_negative": lambda r1, r2: r1 < 0 and r2 < 0,
    "same_parity": lambda r1, r2: (r1 % 2) == (r2 % 2),
    "r1_even": lambda r1, r2: r1 % 2 == 0,
    "r2_even": lambda r1, r2: r2 % 2 == 0,
    "r2_positive": lambda r1, r2: r2 > 0,
    "both_even": lambda r1, r2: r1 % 2 == 0 and r2 % 2 == 0,
    "abs_sum_at_least_4": lambda r1, r2: abs(r1) + abs(r2) >= 4,
    "abs_sum_at_least_5": lambda r1, r2: abs(r1) + abs(r2) >= 5,
    "abs_diff_at_least_2": lambda r1, r2: abs(abs(r1) - abs(r2)) >= 2,
    "abs_diff_at_most_1": lambda r1, r2: abs(abs(r1) - abs(r2)) <= 1,
    "max_at_least_2": lambda r1, r2: max(r1, r2) >= 2,
    "min_at_most_minus_2": lambda r1, r2: min(r1, r2) <= -2,
    "exactly_one_large_abs": lambda r1, r2: (abs(r1) >= 2) ^ (abs(r2) >= 2),
    "at_least_one_positive": lambda r1, r2: r1 > 0 or r2 > 0,
    "at_least_one_negative": lambda r1, r2: r1 < 0 or r2 < 0,
    "r1_odd": lambda r1, r2: r1 % 2 != 0,
}

@dataclass(frozen=True)
class BinaryResponse:
    probe_1: Label
    probe_2: Label
    probe_3: Label
    probe_4: Label

    def as_tuple(self) -> tuple[str, str, str, str]:
        return (
            _coerce_label(self.probe_1, "probe_1"),
            _coerce_label(self.probe_2, "probe_2"),
            _coerce_label(self.probe_3, "probe_3"),
            _coerce_label(self.probe_4, "probe_4"),
        )

class KaggleExecutionError(RuntimeError):
    pass

def _label_for_rule(rule_id: str, point: tuple[int, int]) -> str:
    predicate = RULES[rule_id]
    return "type_a" if predicate(point[0], point[1]) else "type_b"


In [ ]:
def normalize_binary_response(response: object) -> tuple[str, ...] | None:
    if response is None:
        return None
    if isinstance(response, BinaryResponse):
        return response.as_tuple()
    if isinstance(response, str):
        return _parse_text_response(response)
    coerced = _try_coerce(response)
    return coerced.as_tuple() if coerced is not None else None

def _parse_text_response(text: str) -> tuple[str, ...] | None:
    tokens = tuple(
        item.strip().lower()
        for item in text.strip().strip("`").replace("\n", ",").split(",")
        if item.strip()
    )
    return _norm_labels(tokens)

def _try_coerce(response: object) -> BinaryResponse | None:
    if isinstance(response, dict):
        values = response
    elif hasattr(response, "__getitem__") and hasattr(response, "keys"):
        try:
            values = {field: response[field] for field in PROBE_FIELDS}
        except (KeyError, TypeError):
            return None
    elif all(hasattr(response, field) for field in PROBE_FIELDS):
        values = {field: getattr(response, field) for field in PROBE_FIELDS}
    else:
        return None
    try:
        labels = tuple(Label(_coerce_label(values[field], field)) for field in PROBE_FIELDS)
    except (KeyError, TypeError):
        return None
    return BinaryResponse(*labels)

def _coerce_label(value: object, field: str) -> str:
    try:
        return _normalize_label(value)
    except ValueError as exc:
        raise ValueError(f"invalid field {field}: {value!r}") from exc

def _norm_labels(labels: tuple[str, ...] | tuple[Label, ...] | None) -> tuple[str, ...] | None:
    if labels is None:
        return None
    try:
        normalized = tuple(_normalize_label(label.value if isinstance(label, Label) else label) for label in labels)
    except ValueError:
        return None
    return normalized if len(normalized) == PROBE_COUNT else None

def _normalize_label(value: object) -> str:
    if isinstance(value, Enum):
        value = value.value
    elif hasattr(value, "value"):
        value = value.value
    normalized = str(value).strip().lower()
    if normalized not in _ALLOWED_LABELS:
        raise ValueError(f"unknown label: {value}")
    return normalized


In [ ]:
def score_episode(
    predictions: tuple[str, ...] | tuple[Label, ...] | None,
    targets: tuple[str, ...] | tuple[Label, ...],
) -> dict[str, object]:
    norm_targets = _norm_labels(targets)
    if norm_targets is None:
        raise ValueError(f"targets must contain exactly {PROBE_COUNT} valid labels")
    norm_predictions = _norm_labels(predictions)
    if norm_predictions is None:
        norm_predictions = tuple("" for _ in range(PROBE_COUNT))
        numerator = 0
    else:
        numerator = sum(pred == target for pred, target in zip(norm_predictions, norm_targets))
    return {
        "numerator": numerator,
        "denominator": PROBE_COUNT,
        "predictions": tuple(norm_predictions),
    }

@kbench.task(store_task=False)
def run_binary_task(
    llm: object,
    turns: list[str],
    final_probe_targets: tuple[str, ...] | tuple[Label, ...],
) -> dict[str, object]:
    if len(turns) != TURN_COUNT:
        raise KaggleExecutionError(f"expected {TURN_COUNT} turns, received {len(turns)}")
    try:
        llm.prompt(turns[0])
        llm.prompt(turns[1])
        response = llm.prompt(turns[2], schema=BinaryResponse)
    except Exception as exc:
        raise KaggleExecutionError("llm.prompt failed") from exc

    try:
        normalized = normalize_binary_response(response)
    except ValueError as exc:
        raise KaggleExecutionError(f"invalid binary response: {exc}") from exc

    if normalized is None:
        raise KaggleExecutionError(f"unscoreable response of type {type(response).__name__}")
    return score_episode(normalized, final_probe_targets)


In [ ]:
import json

def load_selected_rows() -> list[dict[str, object]]:
    return _load_rows(ROWS_PATH)

def attach_selected_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    if EVAL_SPLIT == "public":
        return rows
    return _attach_private_scoring(rows)

def _count_turn_examples(turn: str) -> int:
    return sum(EXAMPLE_LINE_RE.match(line.strip()) is not None for line in turn.splitlines())

def _count_turn_probes(turn: str) -> int:
    return sum(PROBE_LINE_RE.match(line.strip()) is not None for line in turn.splitlines())

def _validate_turn(turn: str, episode_id: str, turn_index: int) -> None:
    if not turn.startswith(f"{TURN_HEADER_PREFIX}{episode_id}. Turn {turn_index} of 3."):
        raise ValueError(f"episode {episode_id}: malformed header for turn {turn_index}")
    if turn_index in {1, 2}:
        if _count_turn_examples(turn) != 4:
            raise ValueError(f"episode {episode_id}: turn {turn_index} must contain 4 labeled examples")
    else:
        if _count_turn_probes(turn) != PROBE_COUNT:
            raise ValueError(f"episode {episode_id}: decision turn must contain {PROBE_COUNT} probes")
        if OUTPUT_INSTRUCTION not in turn:
            raise ValueError(f"episode {episode_id}: decision turn must use the fixed output instruction")

def _validate_row(row: dict[str, object]) -> None:
    episode_id = str(row["episode_id"])
    analysis = row["analysis"]
    turns = row["inference"]["turns"]

    if analysis["faculty_id"] != FACULTY_ID:
        raise ValueError(f"episode {episode_id}: unsupported faculty_id {analysis['faculty_id']!r}")
    if analysis["group_id"] not in ALLOWED_GROUPS:
        raise ValueError(f"episode {episode_id}: unsupported group_id {analysis['group_id']!r}")
    if analysis["shift_mode"] != GROUP_SHIFT_MODES[analysis["group_id"]]:
        raise ValueError(f"episode {episode_id}: shift_mode mismatch for group {analysis['group_id']!r}")
    if not isinstance(turns, list) or len(turns) != TURN_COUNT:
        raise ValueError(f"episode {episode_id}: expected exactly {TURN_COUNT} turns")
    for turn_index, turn in enumerate(turns, start=1):
        if not isinstance(turn, str) or not turn.strip():
            raise ValueError(f"episode {episode_id}: turn {turn_index} must be a non-empty string")
        _validate_turn(turn, episode_id, turn_index)

def _validate_batch(rows: list[dict[str, object]]) -> None:
    expected_total = EXPECTED_PUBLIC_EPISODE_COUNT if EVAL_SPLIT == "public" else EXPECTED_PRIVATE_EPISODE_COUNT
    if len(rows) != expected_total:
        raise RuntimeError(
            f"benchmark dataset mismatch: expected {expected_total} episodes for split {EVAL_SPLIT!r}, but loaded {len(rows)} from {ROWS_PATH}."
        )

    counts = Counter(str(row["analysis"]["group_id"]) for row in rows)
    expected_per_group = EXPECTED_EPISODES_PER_GROUP[EVAL_SPLIT]
    if set(counts) != ALLOWED_GROUPS:
        raise RuntimeError(f"benchmark groups mismatch: expected {sorted(ALLOWED_GROUPS)}, got {sorted(counts)}")
    bad_counts = {group_id: count for group_id, count in counts.items() if count != expected_per_group}
    if bad_counts:
        raise RuntimeError(f"benchmark per-group counts mismatch for split {EVAL_SPLIT!r}: {bad_counts}")

def _normalize_final_probe_targets(values: object) -> tuple[str, ...]:
    if not isinstance(values, (list, tuple)):
        raise ValueError("final_probe_targets must be a list or tuple")
    labels = _norm_labels(tuple(values))
    if labels is None:
        raise ValueError(f"final_probe_targets must contain exactly {PROBE_COUNT} valid labels")
    return labels

def _load_rows(path: Path) -> list[dict[str, object]]:
    rows = json.loads(path.read_text("utf-8"))
    scoped_rows: list[dict[str, object]] = []
    for row in rows:
        analysis = row.get("analysis")
        inference = row.get("inference")
        scoring = row.get("scoring")
        if not isinstance(analysis, dict):
            raise RuntimeError(f"row {row.get('episode_id')} is missing analysis metadata")
        if not isinstance(inference, dict):
            raise RuntimeError(f"row {row.get('episode_id')} is missing inference fields")
        if EVAL_SPLIT == "public":
            if not isinstance(scoring, dict):
                raise RuntimeError(f"row {row.get('episode_id')} is missing scoring fields")
        elif scoring is not None:
            raise RuntimeError(f"row {row.get('episode_id')} leaks scoring fields in the private split")

        normalized_row = {
            "episode_id": row["episode_id"],
            "inference": {"turns": list(inference["turns"])},
            "analysis": {
                "faculty_id": analysis["faculty_id"],
                "group_id": analysis["group_id"],
                "transition_family_id": analysis["transition_family_id"],
                "initial_rule_id": analysis["initial_rule_id"],
                "shift_rule_id": analysis["shift_rule_id"],
                "shift_mode": analysis["shift_mode"],
            },
        }
        if EVAL_SPLIT == "public":
            normalized_row["scoring"] = {"final_probe_targets": _normalize_final_probe_targets(scoring["final_probe_targets"])}
        _validate_row(normalized_row)
        scoped_rows.append(normalized_row)

    _validate_batch(scoped_rows)
    return scoped_rows

def _load_private_answer_key(path: Path) -> dict[str, dict[str, object]]:
    payload = json.loads(path.read_text("utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("private answer key payload must be a JSON object")
    if payload.get("version") != "cogflex_multi_turn":
        raise RuntimeError("private answer key has an unsupported version")
    if payload.get("split") != "private":
        raise RuntimeError("private answer key must declare split='private'")
    episodes = payload.get("episodes")
    if not isinstance(episodes, list):
        raise RuntimeError("private answer key must expose an episodes list")
    answer_map: dict[str, dict[str, object]] = {}
    for answer in episodes:
        if not isinstance(answer, dict):
            raise RuntimeError("private answer key episodes must be objects")
        episode_id = str(answer.get("episode_id"))
        if episode_id in answer_map:
            raise RuntimeError(f"private answer key duplicates episode_id {episode_id}")
        answer_map[episode_id] = answer
    return answer_map

def _attach_private_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    if PRIVATE_ANSWER_KEY_PATH is None:
        raise RuntimeError(
            "Private split requires an external answer key. "
            f"Set {PRIVATE_ANSWER_KEY_PATH_ENV_VAR} to a local private_answer_key.json path."
        )
    if not PRIVATE_ANSWER_KEY_PATH.exists():
        raise FileNotFoundError(f"Private answer key not found: {PRIVATE_ANSWER_KEY_PATH}")

    answer_map = _load_private_answer_key(PRIVATE_ANSWER_KEY_PATH)
    scored_rows: list[dict[str, object]] = []
    seen_ids: set[str] = set()
    for row in rows:
        episode_id = str(row["episode_id"])
        answer = answer_map.get(episode_id)
        if answer is None:
            raise RuntimeError(f"private answer key is missing episode_id {episode_id}")
        for key in ("faculty_id", "group_id", "transition_family_id", "initial_rule_id", "shift_rule_id", "shift_mode"):
            if answer.get(key) != row["analysis"][key]:
                raise RuntimeError(f"private answer key {key} mismatch for episode {episode_id}")
        scored_rows.append(
            {
                "episode_id": row["episode_id"],
                "inference": {"turns": list(row["inference"]["turns"])},
                "scoring": {"final_probe_targets": _normalize_final_probe_targets(answer.get("final_probe_targets"))},
                "analysis": dict(row["analysis"]),
            }
        )
        seen_ids.add(episode_id)
    extra_ids = sorted(set(answer_map) - seen_ids)
    if extra_ids:
        raise RuntimeError(f"private answer key contains unknown episode_ids: {extra_ids}")
    return scored_rows

leaderboard_rows = load_selected_rows()
scored_rows = attach_selected_scoring(leaderboard_rows)
loaded_episode_count = len(scored_rows)
df = pd.DataFrame(
    [
        {"turns": row["inference"]["turns"], "final_probe_targets": row["scoring"]["final_probe_targets"]}
        for row in scored_rows
    ]
)


In [ ]:
def summarize_ruleshift_benchmark(runs, episode_count: int) -> dict[str, int | float]:
    if not runs:
        raise RuntimeError("evaluation produced no successful runs")
    results_df = runs.as_dataframe()
    if len(results_df) != episode_count:
        raise RuntimeError(f"evaluation completed {len(results_df)} of {episode_count} benchmark episodes")
    numerator = int(results_df.result.map(lambda item: int(item["numerator"])).sum())
    denominator = int(results_df.result.map(lambda item: int(item["denominator"])).sum())
    if denominator == 0:
        raise RuntimeError("evaluation produced a zero denominator")
    return {
        "score": numerator / denominator,
        "numerator": numerator,
        "denominator": denominator,
        "episodes": episode_count,
    }

@kbench.task(
    name="ruleshift_cogflex_binary",
    description="Multi-turn cognitive-flexibility benchmark with explicit, latent, reversal, and context-bound rule shifts.",
)
def ruleshift_cogflex_binary(llm) -> float:
    runs = run_binary_task.evaluate(llm=[llm], evaluation_data=df)
    summary = summarize_ruleshift_benchmark(runs, len(df))
    print(json.dumps(summary, indent=2))
    return summary["score"]


In [ ]:
def _parse_probe_specs(turn: str) -> list[dict[str, object]]:
    specs: list[dict[str, object]] = []
    for line in turn.splitlines():
        match = PROBE_LINE_RE.match(line.strip())
        if match is None:
            continue
        specs.append(
            {
                "index": int(match.group("index")),
                "r1": int(match.group("r1")),
                "r2": int(match.group("r2")),
                "context": match.group("context"),
            }
        )
    return specs

def _previous_rule_prediction(row: dict[str, object]) -> tuple[str, ...]:
    return tuple(
        _label_for_rule(row["analysis"]["initial_rule_id"], (probe["r1"], probe["r2"]))
        for probe in _parse_probe_specs(row["inference"]["turns"][2])
    )

def _context_agnostic_prediction(row: dict[str, object]) -> tuple[str, ...]:
    return tuple(
        _label_for_rule(row["analysis"]["shift_rule_id"], (probe["r1"], probe["r2"]))
        for probe in _parse_probe_specs(row["inference"]["turns"][2])
    )

def build_ruleshift_diagnostics(runs, rows: list[dict[str, object]]) -> dict[str, object]:
    results_df = runs.as_dataframe().reset_index(drop=True)
    if len(results_df) != len(rows):
        raise RuntimeError(f"diagnostics expected {len(rows)} runs, but received {len(results_df)}")

    records: list[dict[str, object]] = []
    perseverative_errors = 0
    perseverative_total = 0
    context_confusion_errors = 0
    context_confusion_total = 0

    for row, result in zip(rows, results_df.result):
        predictions = tuple(result["predictions"])
        targets = tuple(row["scoring"]["final_probe_targets"])
        previous = _previous_rule_prediction(row)
        context_agnostic = _context_agnostic_prediction(row) if row["analysis"]["group_id"] == "context_switch" else None
        misses = [pred != target for pred, target in zip(predictions, targets)]
        miss_pattern = "".join("1" if miss else "0" for miss in misses)
        for idx, miss in enumerate(misses):
            if not miss:
                continue
            if row["analysis"]["group_id"] == "context_switch":
                context_confusion_total += 1
                if context_agnostic is not None and predictions[idx] == context_agnostic[idx]:
                    context_confusion_errors += 1
            else:
                perseverative_total += 1
                if predictions[idx] == previous[idx]:
                    perseverative_errors += 1
        records.append(
            {
                "episode_id": row["episode_id"],
                "group_id": row["analysis"]["group_id"],
                "shift_mode": row["analysis"]["shift_mode"],
                "numerator": int(result["numerator"]),
                "denominator": int(result["denominator"]),
                "miss_pattern": miss_pattern,
            }
        )

    diagnostics_df = pd.DataFrame(records)
    diagnostics_df["accuracy"] = diagnostics_df["numerator"] / diagnostics_df["denominator"]

    def _aggregate(column: str) -> list[dict[str, object]]:
        grouped = (
            diagnostics_df.groupby(column, dropna=False)
            .agg(episodes=("episode_id", "count"), numerator=("numerator", "sum"), denominator=("denominator", "sum"))
            .reset_index()
        )
        grouped["accuracy"] = grouped["numerator"] / grouped["denominator"]
        return grouped.sort_values(["accuracy", column], ascending=[False, True]).to_dict("records")

    miss_pattern = (
        diagnostics_df.groupby("miss_pattern")
        .agg(episodes=("episode_id", "count"))
        .reset_index()
        .sort_values("miss_pattern")
        .to_dict("records")
    )

    return {
        "overall": {
            "episodes": int(diagnostics_df.shape[0]),
            "numerator": int(diagnostics_df["numerator"].sum()),
            "denominator": int(diagnostics_df["denominator"].sum()),
            "accuracy": float(diagnostics_df["numerator"].sum() / diagnostics_df["denominator"].sum()),
        },
        "group_accuracy": _aggregate("group_id"),
        "shift_mode_accuracy": _aggregate("shift_mode"),
        "perseveration_rate": float(perseverative_errors / perseverative_total) if perseverative_total else 0.0,
        "context_confusion_rate": float(context_confusion_errors / context_confusion_total) if context_confusion_total else 0.0,
        "miss_pattern": miss_pattern,
    }


## Official Entry Point

In [ ]:
# ruleshift_benchmark_v1_binary.run(kbench.llm)

In [ ]:
%choose ruleshift_cogflex_binary
